# Taller de IA · El problema del viajante (TSP)
## A* con MST · Hill Climbing vs. Simulated Annealing · Algoritmo genético · RBFS

**Versión del estudiante — enunciado y pseudocódigo, sin implementaciones.**

**Duración propuesta:** dos sesiones. **Modalidad:** parejas. **Entrega:** este mismo cuaderno, completado con las implementaciones propias, pruebas, figuras, resultados y análisis. El docente puede ajustar plazo y modalidad.

**Pregunta central:** ¿qué se gana y qué se pierde al cambiar garantía de optimalidad por rapidez, diversidad o menor consumo de memoria?

Instancias obligatorias: **att48, berlin52, brazil58 y gr48**. Se trabajará también con subinstancias pequeñas para verificar los métodos exactos. Instancias de ampliación: gr17, burma14 y eil51.

Este cuaderno contiene únicamente celdas Markdown. Los bloques etiquetados como `text` son pseudocódigo: no son Python ni MATLAB y no se ejecutan. El estudiante agregará sus propias celdas de código donde se indique.


## 1. Punto de partida y lectura crítica

La página [TSP Solve](https://sites.google.com/view/tsp-solve/homepage) presenta A*, Hill Climbing y RBFS en MATLAB. Este taller amplía el alcance con recocido simulado y un algoritmo genético. Los procedimientos que siguen son una formulación didáctica propia; no constituyen una traducción del código MATLAB.

Antes de programar, revise estas dos diferencias:

1. Intercambiar las posiciones de dos ciudades es **swap**. El movimiento **2-opt** elimina dos aristas y reconecta el ciclo invirtiendo un segmento. No son movimientos equivalentes en general.
2. En TSP, llegar a la misma ciudad por rutas parciales diferentes no significa llegar al mismo estado: importa también el conjunto de ciudades pendientes. Las reglas de descarte deben respetar esta información.

La página reporta resultados no óptimos para algunas ejecuciones de su variante denominada A* de grafo. Eso motiva una auditoría de estados y reglas de descarte; no demuestra que A* correctamente formulado pierda su garantía de optimalidad.

**Actividad de lectura:** identifique qué código del sitio realiza swap y explique qué debería cambiar para implementar 2-opt. Examine qué información necesita la clave de un estado de A*. No use los tiempos publicados como objetivos de rendimiento de su implementación.


## 2. Objetivos y reglas del trabajo

Al finalizar, deberá poder:

- Modelar el TSP como búsqueda de rutas parciales y como optimización de permutaciones completas.
- Construir y justificar una heurística basada en árbol de expansión mínima (MST).
- Implementar A*, RBFS, Hill Climbing, Simulated Annealing y un algoritmo genético con operadores válidos para permutaciones.
- Diseñar experimentos reproducibles y distinguir solución válida, mejor solución observada y optimalidad demostrada.
- Interpretar convergencia, brecha al óptimo, memoria y costo computacional.

**Implementación:** escriba los cinco algoritmos desde cero. Puede usar herramientas para arreglos, gráficas, tablas, lectura de archivos y medición. Se permite `tsplib95` para cargar datos y obtener pesos oficiales. Implemente el MST mediante Prim o Kruskal; no delegue las búsquedas ni la optimización a un solver, SimpleAI, DEAP o una biblioteca equivalente en la comparación principal.

Los nombres sugeridos de componentes e identificadores están en inglés; las explicaciones y conclusiones se entregan en español. Toda ayuda externa deberá declararse. Cada integrante debe poder explicar y modificar su implementación durante la sustentación.


## 3. Banco de problemas TSPLIB

| Instancia | Ciudades | Tipo de pesos | Óptimo oficial del ciclo | Papel en el taller |
|---|---:|---|---:|---|
| att48 | 48 | ATT: distancia pseudoeuclidiana | 10628 | Detectar errores de distancia y redondeo |
| berlin52 | 52 | EUC_2D: distancia euclidiana entera | 7542 | Relacionar rutas con una representación geométrica |
| brazil58 | 58 | EXPLICIT: pesos de una matriz | 25395 | Separar optimización de coordenadas |
| gr48 | 48 | EXPLICIT: pesos de una matriz | 5046 | Comparar dos problemas del mismo tamaño y estructura diferente |
| gr17 · ampliación | 17 | EXPLICIT | 2085 | Exploración exacta intermedia, bajo límites |
| burma14 · ampliación | 14 | GEO | 3323 | Estudiar un tipo de distancia adicional |
| eil51 · ampliación | 51 | EUC_2D | 426 | Generalización de métodos aproximados |

Fuentes: [documentación TSPLIB](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp95.pdf), [óptimos oficiales](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/TSP-BEST.html) y [repositorio oficial de archivos](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/tspindex.html).

Los valores son unidades del problema, no necesariamente kilómetros. Para las cuatro instancias obligatorias use los archivos TSP, no archivos homónimos de VRP. Descomprima los `.tsp.gz` antes de leerlos. El cuaderno proporciona enlaces; no contiene los archivos de datos.

**Descargas:** [att48](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/att48.tsp.gz) · [berlin52](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/berlin52.tsp.gz) · [brazil58](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/brazil58.tsp.gz) · [gr48](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/gr48.tsp.gz).


### 3.1. Preparación y auditoría de los datos

Lea `NAME`, `TYPE`, `DIMENSION`, `EDGE_WEIGHT_TYPE` y, cuando exista, `EDGE_WEIGHT_FORMAT`. Reconstruya una matriz simétrica de costos `D`. No presuponga que la numeración de nodos coincide con las posiciones de una lista: conserve una correspondencia explícita entre identificadores y posiciones.

Se recomienda obtener cada peso mediante la interfaz de `tsplib95`, que distingue pesos explícitos y funciones de distancia. La biblioteca no incluye las instancias. Consulte su [documentación de uso](https://tsplib95.readthedocs.io/en/stable/pages/usage.html).

```text
PROCEDURE LoadInstance(file)
    problem ← leer archivo TSPLIB descomprimido
    nodeIds ← lista ordenada de identificadores reales
    PARA cada par de posiciones i, j
        D[i,j] ← peso oficial entre nodeIds[i] y nodeIds[j]
    verificar dimensión, simetría, diagonal cero y pesos no negativos
    devolver metadatos, nodeIds, D y datos de dibujo disponibles
```

**Entregue:** una ficha por instancia, tres pesos calculados o consultados manualmente y una comprobación automática independiente del costo de una ruta. Si usa un archivo `.opt.tour` oficial disponible, verifique su validez y costo; no lo use como población inicial ni como ruta inicial.

En problemas EXPLICIT, los datos de dibujo, si existen, sirven para visualizar, no para recalcular pesos. Si no hay coordenadas, use una matriz de calor y un dibujo circular de la ruta, etiquetado como disposición esquemática.


In [ ]:
!pip install -q tsplib95

### 3.2. Dos reglas de distancia que deben comprender

Para berlin52 (EUC_2D), con $r=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}$:

$$d(i,j)=\lfloor r+0.5\rfloor.$$

Para att48 (ATT):

$$r=\sqrt{\frac{(x_i-x_j)^2+(y_i-y_j)^2}{10}},\qquad t=\lfloor r+0.5\rfloor,$$

$$d(i,j)=\begin{cases}t+1&\text{si }t<r,\\t&\text{en otro caso.}\end{cases}$$

Para brazil58 y gr48 se utilizan los pesos del archivo, respetando su formato matricial. No reemplace esos pesos por distancias euclidianas. Estas reglas corresponden a la especificación TSPLIB; no aplique el redondeo de empates de un lenguaje sin comprobarlo.

**Ejercicio:** calcule un peso ATT y uno EUC_2D a mano y compárelos con el lector. Explique por qué una ruta aparentemente mejor que el óptimo oficial obliga primero a revisar distancias, cierre del ciclo y validez. La [FAQ oficial](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/TSPFAQ.html) documenta errores frecuentes de este tipo.


## 4. Modelo común y arquitectura de su solución

Sea $V$ el conjunto de ciudades y $s$ una ciudad inicial fija. Una ruta completa es una permutación $\pi=(s,\pi_1,\ldots,\pi_{n-1})$. El origen no se repite en la representación; el costo incluye el regreso:

$$C(\pi)=\sum_{i=0}^{n-2}d(\pi_i,\pi_{i+1})+d(\pi_{n-1},s).$$

| Componente sugerido | Responsabilidad | Evidencia exigida |
|---|---|---|
| `TSPInstance` | Pesos, metadatos e identificadores | Fichas y auditoría |
| `Tour` | Permutación, validez y costo | Casos de cierre y duplicados |
| `SearchNode` | Estado parcial, costo y reconstrucción | Traza de búsqueda |
| `MSTHeuristic` | MST y conexiones de extremos | Justificación y casos frontera |
| `AStarSolver`, `RBFSSolver` | Búsqueda de rutas parciales | Pruebas exactas pequeñas |
| `HillClimbingSolver`, `AnnealingSolver`, `GeneticSolver` | Optimización de rutas completas | Curvas y repeticiones |
| `ExperimentRunner`, `RunResult` | Presupuestos, semillas y medición | Tablas reproducibles |

Puede seguir esta estructura orientada a objetos o módulos con responsabilidades equivalentes. No duplique la lógica de costo en cada algoritmo.

**Contrato de salida:** instancia, algoritmo, parámetros, semilla, ruta válida o ausencia de ruta, costo, tiempo, memoria máxima, motivo de parada y condición de optimalidad. Una ruta inicial de respaldo se identifica como tal; no cuenta como una ruta descubierta por A* o RBFS.


## 5. Rutas parciales para A* y RBFS

Un nodo conserva `last`, `remaining`, `g`, `parent` y `closed`. `g` mide solamente aristas ya recorridas. `closed` distingue el regreso final al origen.

- Raíz: `last = s`, `remaining = V − {s}`, `g = 0`, `closed = false`.
- Extensión: ir desde `last` a una ciudad pendiente y eliminarla de `remaining`.
- Si no quedan pendientes, generar un único sucesor que regresa a `s`, añade ese costo y marca `closed = true`.
- Meta: `closed = true`. Una ruta que ya visitó todas las ciudades pero aún no regresó no es meta.

```text
PROCEDURE Successors(node)
    SI node.closed: devolver lista vacía
    SI node.remaining está vacío
        devolver único nodo cerrado con g = node.g + D[node.last,s]
    successors ← lista vacía
    PARA cada v en node.remaining
        child.last ← v
        child.remaining ← node.remaining − {v}
        child.g ← node.g + D[node.last,v]
        child.parent ← node; child.closed ← false
        añadir child a successors
    devolver successors
```

**Clave de estado:** `(last, remaining, closed)`, con el mismo origen fijo. Dos nodos con esta clave comparten las mismas continuaciones; puede conservarse el de menor `g`. La ciudad, la profundidad o la cantidad de pendientes por sí solas no identifican el estado.

**Actividad:** construya dos rutas parciales que terminen en la misma ciudad y tengan conjuntos pendientes distintos. Muestre por qué fusionarlas puede eliminar la ruta óptima.


## 6. Heurística MST: definición y demostración

Para una ciudad actual $v$ y pendientes $U$:

$$h(v,U)=\begin{cases}
0&\text{si el ciclo está cerrado},\\
d(v,s)&\text{si }U=\varnothing\text{ y falta cerrar},\\
\operatorname{MST}(U)+\min_{u\in U}d(v,u)+\min_{u\in U}d(u,s)&\text{en otro caso.}
\end{cases}$$

Se define $\operatorname{MST}(\varnothing)=\operatorname{MST}(\{u\})=0$. Para un solo pendiente la fórmula calcula exactamente el costo restante. En la raíz, las dos conexiones mínimas pueden seleccionar la misma ciudad; es una relajación válida aunque dé una cota menos ajustada.

**Idea de admisibilidad:** toda continuación conecta la ciudad actual con el primer pendiente, recorre todos los pendientes y termina conectando el último con el origen. Las aristas internas forman un camino que abarca $U$, cuyo costo es al menos el del MST. Cada arista de conexión cuesta al menos el mínimo correspondiente. La suma no sobreestima el costo restante. No se necesita desigualdad triangular para esta justificación en el TSP simétrico de pesos no negativos.

**Tareas:** redacte la demostración con sus palabras; examine además la desigualdad de consistencia $h(n)\leq c(n,n')+h(n')$ para las transiciones del modelo. A* deberá permitir mejores valores de `g` aunque su análisis de consistencia esté incompleto.

**MST no es una ruta TSP:** no impone grado dos ni produce un ciclo que visite una sola vez cada ciudad.


### 6.1. Pseudocódigo de Prim y de la cota

```text
PROCEDURE MSTCost(U)
    SI |U| ≤ 1: devolver 0
    connected ← {una ciudad de U}; total ← 0
    MIENTRAS connected ≠ U
        (a,b) ← arista de menor peso con a en connected y b en U − connected
        total ← total + D[a,b]
        añadir b a connected
    devolver total

PROCEDURE Heuristic(node)
    SI node.closed: devolver 0
    U ← node.remaining
    SI U está vacío: devolver D[node.last,s]
    devolver MSTCost(U) + mínimo D[node.last,u] para u en U
                       + mínimo D[u,s] para u en U
```

El primer bloque expresa Prim de forma conceptual. Una implementación que vuelve a explorar todas las aristas de corte en cada paso puede costar $O(|U|^3)$. Para obtener $O(|U|^2)$ con matriz densa, mantenga para cada ciudad pendiente su mejor costo de conexión y actualícelo al añadir un vértice.

**Experimento opcional:** almacenar `MSTCost(U)` por conjunto de pendientes. Compare tiempo y memoria con y sin caché. Una caché sin límite puede crecer exponencialmente; no utilice una caché global ilimitada al presentar RBFS como búsqueda de memoria reducida.


### 6.2. Microcaso manual: primero comprender, luego programar

Use esta matriz simétrica de cinco ciudades:

| D | A | B | C | D | E |
|---|---:|---:|---:|---:|---:|
| A | 0 | 4 | 6 | 7 | 3 |
| B | 4 | 0 | 2 | 5 | 6 |
| C | 6 | 2 | 0 | 3 | 4 |
| D | 7 | 5 | 3 | 0 | 2 |
| E | 3 | 6 | 4 | 2 | 0 |

Para el prefijo A → B: $g=4$, $U=\{C,D,E\}$, MST de costo $5$, conexión B → U de costo $2$ y U → A de costo $3$. Así, $h=10$ y $f=g+h=14$.

**Complete sin código:**

1. Calcule $g$, $h$ y $f$ de todos los hijos de la raíz A.
2. Muestre el primer empate de la frontera y proponga una regla determinista para resolverlo.
3. Calcule la cota para un solo pendiente y para ningún pendiente.
4. Trace los primeros cambios de límite de RBFS.
5. Evalúe un swap y un 2-opt sobre una misma ruta; explique la diferencia.

Luego enumere las rutas del microcaso con su propio verificador y úselo como oráculo de las implementaciones.


## 7. A* con MST y control de estados repetidos

Priorice por $f=g+h$. Use una cola de prioridad y un desempate estable. La ruta se reconstruye mediante nodos o registros inmutables de padres.

```text
PROCEDURE AStar(root, budget)
    root.f ← root.g + Heuristic(root)
    open ← cola de prioridad que contiene root
    bestG[Key(root)] ← root.g
    MIENTRAS open no esté vacía
        SI se alcanzó budget: devolver INTERRUPTED, sin certificado
        node ← extraer mínimo f de open
        SI node.g > bestG[Key(node)]: continuar
        SI node.closed: devolver ruta reconstruida, OPTIMAL
        registrar expansión de node
        PARA cada child en Successors(node)
            key ← Key(child)
            SI child.g < bestG.get(key, infinito)
                bestG[key] ← child.g
                child.f ← child.g + Heuristic(child)
                insertar child en open
    devolver EXHAUSTED
```

Aquí `bestG` permite insertar una mejor llegada aunque el estado ya se haya expandido; las entradas desactualizadas de la cola se ignoran. No se usa la profundidad como criterio de dominancia. La optimalidad presupone pesos no negativos, heurística admisible, sucesores completos y ausencia de truncamientos.

**Tareas:** implemente la clave con conjuntos inmutables o máscaras de bits; registre nodos generados, expandidos, entradas obsoletas y tamaño máximo de la frontera. Compare `h = 0` y MST sobre los mismos casos pequeños. Cambie el origen y compruebe que el costo óptimo es invariante.

**Ampliación:** incorpore una ruta factible inicial como cota superior. Justifique cada poda con `g+h` y documente cómo certificará el resultado. No denomine A* exacto a una variante que conserve solamente los mejores K nodos.


## 8. RBFS: búsqueda recursiva del mejor primero

RBFS explora el mejor hijo bajo un límite definido por la siguiente alternativa. Conserva valores respaldados de `f` para decidir cuándo regresar y puede regenerar subárboles. Se utilizará **el mismo modelo de estados y la misma cota MST que A***.

```text
PROCEDURE SolveRBFS(root)
    root.f ← root.g + Heuristic(root)
    devolver RBFS(root, infinito)

PROCEDURE RBFS(node, limit)
    SI se alcanzó el presupuesto global: propagar INTERRUPTED
    SI node.closed: devolver FOUND(ruta), node.f
    children ← Successors(node)
    SI children está vacío: devolver FAILURE, infinito
    PARA cada child en children
        child.f ← máximo(child.g + Heuristic(child), node.f)
    REPETIR
        ordenar children por f con desempate estable
        best ← primer hijo
        SI best.f = infinito O best.f > limit
            devolver FAILURE, best.f
        alternative ← f del segundo hijo, o infinito si no existe
        result, backedF ← RBFS(best, mínimo(limit, alternative))
        SI result es INTERRUPTED: propagar INTERRUPTED
        best.f ← backedF
        SI result es FOUND: devolver result, best.f
```

Guía conceptual: [pseudocódigo oficial AIMA para RBFS](https://github.com/aimacode/aima-pseudocode/blob/master/md/Recursive-Best-First-Search.md). La adaptación añade cierre explícito del ciclo y propagación de interrupciones.


### 8.1. Evidencias específicas para RBFS

**Instrumente:** profundidad máxima, expansiones totales, llamadas, regeneraciones y número máximo de nodos simultáneamente retenidos. Describa cómo identifica una regeneración sin alterar indebidamente la memoria del solver; un registro de auditoría externo también consume memoria y debe declararse.

La versión de árbol evita ciclos porque cada extensión reduce pendientes y el último paso cierra el ciclo. No necesita un conjunto global ilimitado de visitados. En el análisis usual RBFS retiene $O(bd)$ registros de nodo. En TSP, $b$ y $d$ crecen con $n$; además, almacenar conjuntos completos en cada registro añade costo. Menor memoria no implica menor tiempo.

Sin interrupciones, con la cota admisible y el árbol finito correctamente generado, se espera un resultado óptimo. Si vence un presupuesto, informe **interrumpido**, no **fracaso definitivo**. No considere un valor respaldado aislado de `f` como certificado automático de brecha tras una interrupción.

**Explique con una traza:** por qué se usa el máximo al asignar `child.f`, qué representa la segunda alternativa y cómo una rama descartada temporalmente puede reaparecer.


## 9. Vecindad común: 2-opt para TSP simétrico

Una ruta es `route[0..n−1]` con origen fijo en la posición cero. Elija dos aristas no adyacentes `(a,b)` y `(c,d)`, donde:

$$a=route[i],\quad b=route[i+1],\quad c=route[j],\quad d=route[(j+1)\bmod n].$$

Use $0\leq i<j<n$, excluya $j=i+1$ y el par $(i,j)=(0,n-1)$. Invierta el segmento de posiciones $i+1$ a $j$, ambos incluidos. La ciudad inicial permanece fija.

$$\Delta=d(a,c)+d(b,d)-d(a,b)-d(c,d).$$

```text
PROCEDURE TwoOptMove(route, i, j)
    verificar que las aristas elegidas no sean adyacentes
    candidate ← copia de route
    invertir candidate desde i+1 hasta j, incluidos
    devolver candidate
```

Para un ciclo simétrico de $n\geq4$ hay $n(n-3)/2$ pares válidos de aristas. La fórmula de $\Delta$ depende de la simetría; no la traslade sin cambios al TSP asimétrico.

**Pruebas:** verifique que cada movimiento conserva la permutación; compare el delta con la diferencia de costos completos; incluya una arista de cierre. Antes de optimizar, implemente la versión que recalcula todo el costo y úsela para validar la incremental.


## 10. Hill Climbing: descenso de costo con mejor mejora

Se mantendrá el nombre habitual Hill Climbing, aunque se minimice el costo. La variante principal recorre todo el vecindario 2-opt y elige la mejor mejora estricta.

```text
PROCEDURE HillClimbing(initialTour, budget)
    current ← initialTour; registrar costo inicial
    REPETIR
        bestMove ← ninguno; bestDelta ← 0
        PARA cada par válido de aristas de current
            SI se alcanzó budget: devolver current, INTERRUPTED
            delta ← evaluar el movimiento 2-opt
            SI delta < bestDelta
                bestDelta ← delta; bestMove ← par actual
        SI bestMove es ninguno: devolver current, LOCAL_OPTIMUM
        current ← aplicar bestMove a current
        registrar costo y trabajo acumulado
```

Con mejoras estrictas y rutas finitas, termina en un óptimo local del vecindario examinado. Si se interrumpe una exploración, no se ha demostrado optimalidad local. No aceptar empates evita recorridos indefinidos entre soluciones de igual costo.

**Variantes requeridas:** una ejecución con mejor mejora y una versión con reinicios aleatorios que conserva la mejor ruta global. Los reinicios comparten el presupuesto de la ejecución. Compare además primera mejora en una subinstancia; no mezcle sus resultados con la variante principal.

**Pregunta:** ¿una ruta sin cruces visibles necesariamente es óptima? Discuta el caso geométrico y el de una matriz explícita.


## 11. Simulated Annealing: aceptar temporalmente empeoramientos

Con $\Delta=C(candidate)-C(current)$, acepte siempre una mejora y un empate; para $\Delta>0$, acepte con probabilidad:

$$p=\exp(-\Delta/T),\qquad T>0.$$

La ruta actual y la mejor ruta observada son objetos distintos. Devuelva la mejor observada, pues la actual puede haber empeorado.

```text
PROCEDURE SimulatedAnnealing(initialTour, T0, Tmin, alpha, steps, budget)
    current ← initialTour; best ← copia de current; T ← T0
    MIENTRAS T > Tmin Y quede presupuesto
        REPETIR hasta steps propuestas o hasta agotar presupuesto
            elegir uniformemente un par válido de aristas
            delta ← evaluar ese movimiento 2-opt
            SI delta ≤ 0 O Uniform(0,1) < exp(−delta/T)
                current ← aplicar movimiento
                SI C(current) < C(best): best ← copia de current
            registrar costo actual, mejor costo, T y aceptación
        T ← alpha × T
    devolver best y motivo de parada
```

Use $0<\alpha<1$ y $T_{min}>0$. Calibre $T_0$ con deltas positivos de movimientos de prueba y una aceptación inicial deseada $p_0$: $T_0=-\operatorname{mediana}(\Delta^+)/\ln(p_0)$. Si no hay deltas positivos, explique su alternativa. Declare y contabilice el trabajo de calibración.

**Diseño:** compare $\alpha\in\{0.90,0.97,0.995\}$ y $p_0\in\{0.5,0.8\}$ en datos de calibración pequeños; fije la configuración antes de la evaluación principal. El recocido geométrico finito no garantiza encontrar el óptimo global.


### 11.1. Comparación controlada HC vs. SA

Para cada instancia y semilla use **la misma ruta inicial** y la misma vecindad. Asigne generadores aleatorios independientes a inicialización y a decisiones de cada algoritmo. Esto evita que cambios de implementación alteren inadvertidamente las rutas iniciales.

Presente dos comparaciones:

1. HC con una sola ruta vs. SA: el efecto de aceptar empeoramientos.
2. HC con reinicios vs. SA: uso de un presupuesto global compartido.

Grafique el mejor costo acumulado frente al número de movimientos evaluados y, por separado, frente al tiempo. Incluya una curva de costo actual de SA y su temperatura. Extender una curva constante tras la parada solo se permite si se identifica como conservación del resultado, no como evaluaciones ejecutadas.

**Preguntas:** ¿SA mejora siempre a HC? ¿Qué ocurre con temperatura inicial demasiado baja? ¿Cómo afecta el enfriamiento a la aceptación? ¿Una mejora final compensa necesariamente el trabajo adicional?


## 12. Algoritmo genético para permutaciones

Cada cromosoma contiene todas las ciudades excepto el origen fijo. La decodificación antepone el origen y cierra implícitamente el ciclo. Use torneo por costo mínimo, cruce de orden OX, mutación por inversión y elitismo explícito.

```text
PROCEDURE GeneticAlgorithm(populationSize, eliteCount, pc, pm, budget)
    population ← populationSize permutaciones aleatorias válidas
    evaluar población y registrar best
    MIENTRAS quede presupuesto para la generación acordada
        nextPopulation ← copias de eliteCount mejores individuos
        MIENTRAS nextPopulation no esté completa
            parent1, parent2 ← torneos por menor costo
            SI Uniform(0,1) < pc
                children ← OX(parent1,parent2) y OX(parent2,parent1)
            EN OTRO CASO: children ← copias de los padres
            PARA cada child requerido
                SI Uniform(0,1) < pm: invertir un segmento de child
                verificar permutación, evaluar y añadir child
        population ← nextPopulation
        actualizar best y registrar diversidad
    devolver best y motivo de parada
```

No aplique cruce de un punto convencional a permutaciones: puede duplicar ciudades y omitir otras. `pm` es probabilidad de mutar un individuo, no cada gen. Defina tamaños de torneo, población y élite, políticas de duplicados y desempates.

Configuraciones de calibración sugeridas: población 40 u 80, torneo de tamaño 3, una o dos élites, $p_c=0.9$ y $p_m\in\{0.1,0.3\}$. Son puntos de partida, no parámetros universalmente óptimos.


### 12.1. Cruce OX y diversidad

```text
PROCEDURE OX(parent1, parent2)
    elegir posiciones left ≤ right dentro del cromosoma
    child ← posiciones vacías
    copiar parent1[left..right] en las mismas posiciones de child
    positions ← posiciones vacías de child recorridas circularmente desde right+1
    genes ← parent2 recorrido circularmente desde right+1
    PARA cada gene en genes
        SI gene aún no está en child
            colocar gene en la siguiente posición de positions
    devolver child
```

**Actividad manual:** complete OX para `parent1 = [B,C,D,E,F,G]`, `parent2 = [E,B,G,C,F,D]` y segmento de posiciones 2 a 4 contando desde 1. Trace orden de lectura, ciudades omitidas y posiciones rellenadas.

**Actividad experimental:** mida proporción de individuos únicos y, opcionalmente, diversidad de aristas. Para contar ciclos equivalentes, rote al origen y elija una representación canónica entre una ruta y su reversa; explique cómo evita alterar la evaluación.

**Ampliación separada:** aplique 2-opt a los descendientes y denomine esa variante algoritmo memético. Cuente su trabajo adicional y compárela aparte del GA básico. No atribuya automáticamente al cruce la mejora producida por la búsqueda local.


## 13. Validación antes de las instancias completas

Agregue sus propias celdas con pruebas para:

| Elemento | Caso exigido | Resultado que debe verificarse |
|---|---|---|
| Ruta | Ciudad duplicada u omitida | Se rechaza como inválida |
| Costo | Ruta completa con cierre | Coincide con suma independiente |
| MST | Cero, uno y varios pendientes | Casos frontera y árbol válido |
| Heurística | Prefijos de un caso de 6–8 ciudades | $h$ no supera el residual óptimo enumerado |
| A* | Microcaso y subinstancias pequeñas | Mismo costo que enumeración exhaustiva |
| A* | Dos conjuntos pendientes distintos | No se fusionan por ciudad final |
| RBFS | Un hijo, empate, límite y agotamiento | Retroceso y terminación correctos |
| HC | Vecindario completo sin mejora | Óptimo local verificado |
| SA | Mejor ruta distinta de la actual | Se conserva la mejor observada |
| OX e inversión | Múltiples semillas y cortes frontera | Toda ciudad aparece exactamente una vez |
| Presupuesto | Interrupción forzada | Motivo correcto y ruta válida o ausencia explícita |

Para validar $h$, enumere **continuaciones de cada prefijo**, no solo ciclos desde la raíz. No confunda el costo total óptimo con el costo restante de un nodo.

El oráculo exhaustivo solo se usa en casos pequeños para pruebas. Un resultado igual al óptimo publicado tiene brecha cero frente a la referencia; eso no convierte un algoritmo aproximado en uno con garantía general.


## 14. Protocolo experimental: dos escalas

### Escala A · Comparación de los cinco algoritmos

De cada instancia obligatoria seleccione determinísticamente los primeros $k$ identificadores ordenados, con $k\in\{6,8,10,12\}$. Conserve los pesos originales del subproblema inducido. Registre el listado de nodos, no solo el tamaño. No imponga que esos nodos aparezcan consecutivos en la ruta del problema completo.

- A* y RBFS: una ejecución por caso y origen fijo; repita tres veces para tiempos si el costo lo permite. Una prueba extra con otro origen verifica invariancia del costo.
- Métodos aproximados: diez semillas, de 0 a 9, por caso. Compare sus costos con el óptimo del **subproblema** solo si se certificó mediante A* o RBFS, o mediante enumeración para los tamaños menores.
- Límite propuesto de exactos: 30 segundos, 100000 expansiones o 256 MiB de memoria incremental, lo que ocurra primero. Informe cómo se mide y con qué frecuencia se revisa la memoria.

### Escala B · att48, berlin52, brazil58 y gr48 completas

- HC de una ruta, HC con reinicios, SA y GA: diez semillas por instancia; hasta 30000 evaluaciones de candidatos y un límite adicional de 30 segundos por ejecución.
- A* y RBFS: **un ensayo diagnóstico por instancia**, con los mismos límites de exactos. No se exige que encuentren ruta ni demuestren optimalidad.
- Registre límites efectivos y causas de parada. Puede ajustar los valores por capacidad del equipo, antes de ejecutar la comparación final y aplicándolos de forma uniforme.

**Importante:** el óptimo de una subinstancia no es el óptimo oficial de la instancia completa. Una selección de doce ciudades de att48 no se compara con 10628.


### 14.1. Qué significa presupuesto comparable

Una evaluación local consiste en decidir el costo de un movimiento candidato, aunque se utilice delta. En GA cuente cada cromosoma cuyo costo se calcula, incluida la población inicial; si utiliza caché, registre también candidatos consultados y aciertos. Los individuos élite con costo conservado no requieren reevaluación. No exceda el presupuesto para completar una generación.

En A* y RBFS, expansión, generación de sucesores y cálculo de MST son operaciones distintas. **No equipare una expansión de A* con una evaluación de una ruta de GA.** Use tiempo y memoria para comparar familias, y contadores específicos para explicar el trabajo. El presupuesto de evaluaciones aproxima un esfuerzo común entre métodos locales; no asegura idéntico costo por operación.

Registre costo y tiempo de precarga de datos por separado. Registre toda inicialización y calibración del solver; mantenga las gráficas fuera de la región cronometrada. Para HC frente a SA, conservar una misma ruta inicial no implica excluir su costo sin documentarlo.

Declare CPU, sistema operativo, entorno Jupyter o Colab, versiones de dependencias y medición de memoria. Distinga memoria del proceso completo de memoria adicional atribuible al solver. Si la herramienta solo mide asignaciones de un lenguaje, indíquelo.

**Prueba de reproducibilidad:** reinicie el kernel y ejecute todas sus celdas en orden con los datos locales. Los gráficos deben provenir de los registros; no escriba resultados manuales como si fueran mediciones.


## 15. Métricas y resultados que debe entregar

Para las instancias completas con óptimo oficial $C^*$:

$$\operatorname{gap}(\%)=100\frac{C-C^*}{C^*}.$$

Un gap negativo requiere detenerse y auditar datos, ruta y costo. Para un caso sin referencia certificada, use `N/A`; no reemplace $C^*$ por la mejor ruta del grupo y la llame óptimo.

| Campo por ejecución | Descripción |
|---|---|
| `instance`, `subset`, `algorithm`, `seed`, `parameters` | Identificación completa |
| `tour`, `cost`, `gap_percent` | Resultado y comparación con referencia |
| `elapsed_seconds`, `peak_memory` | Recursos y método de medición |
| `candidate_evaluations` | Trabajo de HC, SA y GA |
| `expanded`, `generated`, `mst_calls` | Trabajo de A* y RBFS |
| `frontier_peak`, `regenerations` | Frontera de A* y recomputación de RBFS |
| `stop_reason`, `optimality_status` | Finalización, límite o evidencia de optimalidad |

**Resumen por instancia y algoritmo:** mejor costo, mediana, media y desviación estándar del costo y gap; mediana del tiempo; tasa de coincidencia con el óptimo de referencia; número de ejecuciones válidas. No agregue los costos crudos de instancias distintas: use gap para comparar calidad entre ellas.

Sin ruta, deje costo y gap como `N/A` e informe cuántas ejecuciones fueron censuradas por límites. Un promedio calculado solo sobre búsquedas que terminaron puede ocultar dificultad; indique siempre su denominador.


### 15.1. Plantillas de tablas

**Complete una tabla de este tipo por instancia:**

| Método | Repeticiones válidas/previstas | Mejor costo | Gap mediano % | Tiempo mediano s | Memoria máxima | Motivo predominante |
|---|---|---|---|---|---|---|
| A* + MST | — | — | — | — | — | — |
| RBFS + MST | — | — | — | — | — | — |
| HC una ruta | — | — | — | — | — | — |
| HC reinicios | — | — | — | — | — | — |
| SA | — | — | — | — | — | — |
| GA básico | — | — | — | — | — | — |

**Para las subinstancias:**

| Instancia | Nodos seleccionados | k | Óptimo certificado | Expansiones A* | Expansiones RBFS | Memoria A* | Memoria RBFS |
|---|---|---|---|---|---|---|---|
| — | — | — | — | — | — | — | — |

No hay resultados precargados: todas las cifras experimentales deben obtenerse con sus implementaciones.


### 15.2. Visualizaciones obligatorias

1. Rutas inicial y final de cada instancia. Si solo hay matriz, utilice dibujo esquemático y matriz de calor; no sugiera que la disposición circular representa geografía real.
2. Mejor costo acumulado frente a evaluaciones para HC, SA y GA, mostrando mediana y dispersión entre semillas.
3. Gap por algoritmo e instancia mediante cajas o puntos individuales.
4. Tiempo y memoria frente al tamaño de subinstancia para A* y RBFS, identificando ejecuciones interrumpidas.
5. Temperatura, costo actual y tasa de aceptación de empeoramientos de SA; defina el tamaño de la ventana de aceptación.
6. Mejor y costo mediano de población, junto con diversidad por generación del GA.

**Animación didáctica:** construya una animación breve de mejoras de una ruta de berlin52, o una secuencia de cuadros incrustada en el cuaderno si el entorno no permite animación. Muestre algoritmo, costo y trabajo acumulado. No genere una figura dentro de cada evaluación durante el cronometraje.


## 16. Discusión: responda con evidencia

1. ¿Por qué A* y RBFS trabajan con rutas parciales mientras HC, SA y GA trabajan con rutas completas?
2. ¿Por qué el MST da una cota y no una solución? ¿Qué información pierde la relajación?
3. ¿Qué sucede si A* cierra estados usando solamente la última ciudad?
4. ¿Por qué devolver el primer nodo con pendientes vacíos, sin cobrar el retorno, puede dar una respuesta incorrecta?
5. ¿En qué casos RBFS redujo memoria y aumentó recomputación? Muestre números y una traza.
6. ¿Qué diferencias encontró entre att48 y gr48, ambas con 48 ciudades?
7. ¿Cuánto aportaron los reinicios frente a aceptar movimientos peores? Analice distribución, no solo la mejor ejecución.
8. ¿Qué signos de convergencia prematura observó en el GA? ¿Cómo se relacionan con la diversidad?
9. ¿Cambió el ranking al comparar tiempo en lugar de evaluaciones?
10. Si dispone de dos segundos para devolver una ruta, ¿qué elegiría según sus datos? ¿Y si necesita demostrar optimalidad?
11. ¿Qué afirmaciones de complejidad o garantía de la página de referencia requieren matices? Distinga propiedades del algoritmo de particularidades de una implementación.
12. ¿Qué conclusión no puede sostener por falta de repeticiones, límites de recursos o selección de datos?

Cada conclusión debe citar una tabla, figura, prueba o traza del cuaderno. Una afirmación como “GA es mejor” sin presupuesto, instancia y dispersión es insuficiente.


## 17. Plan de trabajo y evaluación

| Etapa propuesta | Producto verificable |
|---|---|
| Días 1–2 | Fichas TSPLIB, contrato de ruta y microcaso manual |
| Días 3–5 | MST, A* y RBFS validados en casos pequeños |
| Días 6–8 | Vecindad, HC, SA y GA validados |
| Días 9–11 | Calibración separada y experimentos congelados |
| Días 12–14 | Figuras, discusión, ejecución completa y sustentación |

| Criterio | Peso |
|---|---:|
| Modelo, distancias oficiales y auditoría | 15 % |
| A* + MST: formulación, pruebas y trazas | 15 % |
| RBFS: límites, respaldo de f y medición | 15 % |
| HC vs. SA: operadores y comparación controlada | 15 % |
| GA: permutaciones, OX, mutación y diversidad | 15 % |
| Protocolo, reproducibilidad y análisis de resultados | 15 % |
| Claridad, visualizaciones y sustentación | 10 % |

Las búsquedas exactas que no terminan en las instancias completas **no se penalizan** si son correctas, respetan el protocolo y documentan sus límites. Encontrar una ruta barata no sustituye la corrección ni la calidad experimental.


## 18. Espacios de desarrollo del estudiante

Agregue sus propias celdas de código y Markdown debajo de cada apartado. Mantenga visibles enunciados y pseudocódigos.

### A. Integrantes, entorno y decisiones

*Complete integrantes, equipo, versiones, presupuestos, políticas de desempate y ayuda externa utilizada.*

### B. Datos y verificación del costo

*Agregue lectura de instancias, fichas y pruebas de rutas.*

### C. Modelo de búsqueda, MST y microcaso

*Agregue desarrollo manual, componentes y prueba de admisibilidad.*

### D. A* y RBFS

*Agregue implementaciones, trazas, validación exacta y pruebas de interrupción.*

### E. Vecindad, HC y SA

*Agregue operadores, validación del delta y comparación controlada.*

### F. Algoritmo genético

*Agregue OX manual, operadores, pruebas de permutación y diversidad.*

### G. Experimentos y visualizaciones

*Agregue configuraciones congeladas, registros, tablas, curvas y animación.*

### H. Discusión y conclusiones

*Responda las preguntas y formule conclusiones respaldadas por sus resultados.*


## 19. Lista de entrega

- [ ] El cuaderno se ejecuta desde un kernel reiniciado con archivos de datos locales.
- [ ] Las cuatro instancias obligatorias están auditadas y los pesos son oficiales.
- [ ] Los cinco algoritmos tienen implementación propia y pruebas pertinentes.
- [ ] El costo incluye cierre y todas las rutas entregadas son permutaciones válidas.
- [ ] Hay pruebas pequeñas de optimalidad para A* y RBFS.
- [ ] Se distinguen swap, 2-opt y mutación por inversión.
- [ ] Los registros incluyen semillas, parámetros, presupuestos y causas de parada.
- [ ] Las comparaciones estadísticas incluyen las diez semillas y resultados censurados.
- [ ] Las gráficas proceden de registros y las conclusiones remiten a evidencia.
- [ ] Se entrega el cuaderno con salidas guardadas y los datos o instrucciones de descarga.
- [ ] Los integrantes pueden explicar cualquier fragmento y hacer un cambio durante la sustentación.


## 20. Referencias y alcance de las fuentes

Consulta de fuentes: **5 de octubre de 2026**.

1. Hamdi Altaheri. [TSP Solve — inicio](https://sites.google.com/view/tsp-solve/homepage), [A*](https://sites.google.com/view/tsp-solve/tsp-a-algorithm), [Hill Climbing](https://sites.google.com/view/tsp-solve/tsp-hill-climbing-algorithm) y [RBFS](https://sites.google.com/view/tsp-solve/tsp-rbfs-algorithm). Referencia inicial y objeto de lectura crítica; sus resultados no se presentan como resultados de este taller.
2. Gerhard Reinelt. [TSPLIB95](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/), [documentación](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp95.pdf), [archivos](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/tspindex.html), [óptimos](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/TSP-BEST.html) y [FAQ](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/TSPFAQ.html). Fuente de datos, tipos de distancia y valores de referencia.
3. Russell y Norvig. *Artificial Intelligence: A Modern Approach*, cuarta edición. Capítulo 3: búsqueda informada y memoria limitada; capítulo 4: búsqueda local y optimización. Consulte su ejemplar para el tratamiento teórico.
4. AIMA Code. [Recursive Best First Search](https://github.com/aimacode/aima-pseudocode/blob/master/md/Recursive-Best-First-Search.md). El repositorio identifica ese pseudocódigo como AIMA3e; se usa como apoyo algorítmico, no como reproducción de una figura de la cuarta edición.
5. Desarrolladores de `tsplib95`. [Guía de uso](https://tsplib95.readthedocs.io/en/stable/pages/usage.html). Referencia para lectura y consulta de pesos, sin implementación de los algoritmos solicitados.

Las tablas de resultados están vacías intencionalmente. Este es un enunciado para estudiantes; no incluye soluciones en Python, resultados experimentales inventados ni ejecución de las instancias completas.
